# 1. Synthetic data labelled by construction

Writing thousands of cases with their answers by hand does not scale, and asking an LLM to label them afterwards
inherits its mistakes. `layaft generate` works the other way round: **it decides the answers first and asks an LLM
for a text that has them**, one fixed combination per call, spread evenly.

| | |
|---|---|
| Combinations | every answer of every question, minus the task's `exclude` rules (36 for the helpdesk) |
| Context | `context="..."`, or `context="all"` for the task's contexts file (48 sectors and countries) |
| Signals | each option can demand its own evidence in the facts (a mail asking for a password, an antivirus alert…) |
| Format | the task's fields become a Pydantic model whose JSON Schema constrains the LLM and validates each batch |
| Filters | phrases that give the answer away, the answer's own name, repeated titles and test-set titles |

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")  # data/, tasks/ and runs/ live in the repo root.
from layaft import LayaFT, Task

task = Task.load("helpdesk")

import random
from layaft.data.generate import prompt_for

combos = task.combos()
print(len(combos), "combinations; for example", combos[0])
print(prompt_for(task, combos[-1], "tickets de un banco en España", [], random.Random(0), 5))

## Generate

Any backend: `ollama` (free, local), `openrouter`, `openai`, or `custom` with `base_url=` for any OpenAI-compatible
server. Rows are **appended** to `data/helpdesk.jsonl` as each combination finishes. `filler=` also writes the neutral
documents that long-context training wraps around the cases.

In [ ]:
LayaFT().generate(task, backend="ollama", llm="gemma3:12b", n=36, context="tickets de una clínica en Uruguay")

# The README's dataset (~10,000 cases, ~35 min, US$3):
# LayaFT().generate(task, backend="openrouter", n=216, context="all", parallel=18)
# Filler for long context:
# LayaFT().generate(task, backend="openrouter", n=0, filler=300, context="all")

## What the dataset looks like

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
from layaft.data import io

rows = io.read(io.cases_path(task))
print(len(rows), "cases in", io.cases_path(task))
fig, axes = plt.subplots(1, len(task.questions), figsize=(15, 3.6))
for ax, (qid, q) in zip(axes, task.questions.items()):
    counts = Counter(str(r["answers"][qid]) for r in rows)
    ax.bar(list(counts), list(counts.values()), color="#4fa8f0", edgecolor="#0d0d0d")
    ax.set_title(q.name)
plt.tight_layout()

In [ ]:
for r in random.Random(0).sample(rows, 3):
    print(r["answers"], "\n", task.state(r), "\n")